# BRISC 2025 forensic audit (definitive, Colab)

**Catarina Bota | UCP**

Runs the forensic audit on Colab (Linux) to produce the **canonical** clean dataset (5,032 images) and saves it permanently to Drive.

**Pipeline:**
1. Join `train` + `test` (6,000 imgs)
2. MD5 dedupe -> remove exact duplicates
3. pHash greedy (Hamming <= 5) -> remove near-duplicates
4. Rebuild clean folder at `/content/drive/MyDrive/brisc2025_clean/classification_task/`
5. Save `audit_log.csv` with every removed file and its reason

**Runtime: ~10-15 min** (single-pass, CPU only)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install imagehash -q

In [ ]:
import os, hashlib, shutil, csv
import pandas as pd
from PIL import Image
import imagehash
from tqdm.notebook import tqdm

SRC       = '/content/drive/MyDrive/brisc2025/classification_task'
DST_BASE  = '/content/drive/MyDrive/brisc2025_clean'
DST       = f'{DST_BASE}/classification_task'
LOG_PATH  = f'{DST_BASE}/audit_log.csv'
PHASH_THR = 5

os.makedirs(DST_BASE, exist_ok=True)
print(f'SRC: {SRC}')
print(f'DST: {DST}')

## Stage 1 - enumerate images

Train first, then test (same order as the original delivered notebook).

In [ ]:
all_paths = []
for split in ('train', 'test'):
    for root, _, files in os.walk(f'{SRC}/{split}'):
        for f in files:
            if f.lower().endswith(('.png', '.jpg', '.jpeg')):
                all_paths.append(os.path.join(root, f))

print(f'Total: {len(all_paths)} images')

## Stage 2 - MD5 dedupe (exact duplicates)

In [ ]:
def file_hash(path):
    with open(path, 'rb') as f:
        return hashlib.md5(f.read()).hexdigest()

rows = []
for p in tqdm(all_paths, desc='MD5'):
    rows.append({'path': p, 'hash': file_hash(p)})

df = pd.DataFrame(rows)
df_exact = df.drop_duplicates(subset=['hash'], keep='first')
clean_paths = df_exact['path'].tolist()

md5_removed = [p for p in all_paths if p not in set(clean_paths)]
print(f'Removed (MD5):   {len(md5_removed)}')
print(f'Remaining:       {len(clean_paths)}')

## Stage 3 - pHash greedy (Hamming <= 5)

For each image, compare against all previously kept hashes. If any pair has Hamming <= 5 -> discard (near-duplicate). Otherwise -> keep.

In [ ]:
def get_phash(path):
    try:
        with Image.open(path) as im:
            return imagehash.phash(im.convert('RGB'))
    except Exception:
        return None

phash_of = {p: get_phash(p) for p in tqdm(clean_paths, desc='pHash')}

final = []
kept_hashes = []
removed_phash = []
for p in tqdm(clean_paths, desc='Greedy dedupe'):
    ph = phash_of[p]
    if ph is None:
        continue
    matched = None
    for kp, kh in kept_hashes:
        if ph - kh <= PHASH_THR:
            matched = kp; break
    if matched is not None:
        removed_phash.append((p, matched))
    else:
        final.append(p)
        kept_hashes.append((p, ph))

print(f'Removed (pHash): {len(removed_phash)}')
print(f'FINAL CLEAN:     {len(final)}')

## Stage 4 - copy clean dataset to Drive

In [ ]:
if os.path.exists(DST):
    shutil.rmtree(DST)

for p in tqdm(final, desc='Copy'):
    rel = os.path.relpath(p, SRC)
    dst = os.path.join(DST, rel)
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    shutil.copy2(p, dst)

print(f'Clean folder at: {DST}')

## Stage 5 - audit log + summary

In [ ]:
with open(LOG_PATH, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['action', 'reason', 'removed_path', 'kept_path'])
    for p in md5_removed:
        w.writerow(['remove', 'md5_exact', p, ''])
    for p, kp in removed_phash:
        w.writerow(['remove', f'phash<={PHASH_THR}', p, kp])

print(f'Log: {LOG_PATH}')
print()
print('=' * 50)
print(f'  source           {len(all_paths):6d}')
print(f'  removed (MD5)    {len(md5_removed):6d}')
print(f'  removed (pHash)  {len(removed_phash):6d}')
print(f'  FINAL CLEAN      {len(final):6d}')
print('=' * 50)

print('\nDistribution:')
total = 0
for split in ('train', 'test'):
    sp = f'{DST}/{split}'
    if not os.path.exists(sp):
        continue
    for cls in sorted(os.listdir(sp)):
        n = len(os.listdir(f'{sp}/{cls}'))
        print(f'  {split}/{cls:12s} {n:5d}')
        total += n
print(f'  {"TOTAL":16s} {total:5d}')

## Next steps

Once this notebook finishes, every other notebook (CNN, hybrid, Grad-CAM, EfficientNet) should point to:

```python
CLEAN_DIR = '/content/drive/MyDrive/brisc2025_clean/classification_task'
TRAIN_DIR = f'{CLEAN_DIR}/train'
TEST_DIR  = f'{CLEAN_DIR}/test'
```

This way the full pipeline runs on the **forensically audited** dataset (5,032 images), fixed and reproducible.